In [ ]:
# Canonicalize and sample valid molecules

import os
import pandas as pd
from rdkit import Chem, RDLogger
from tqdm import tqdm

tqdm.pandas()
RDLogger.DisableLog("rdApp.*")

CSV_PATH = "FILE_PATH_HERE"
SMILES_COL = "SMILES"
RENAME_MAP = {"ChemProp (raw)": "binary_activity"}
N_SAMPLE = 50000
SAMPLE_SEED = 42


def canonicalize_smiles(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


def process_file(csv_path, out_path):
    df = pd.read_csv(csv_path)
    missing = [c for c in RENAME_MAP if c not in df.columns]
    if missing:
        print(f"Columns not found, not renamed: {missing}")
    df = df.rename(columns=RENAME_MAP)

    df["canonical_SMILES"] = df[SMILES_COL].progress_apply(canonicalize_smiles)
    valid = df.dropna(subset=["canonical_SMILES"]).copy()
    valid[SMILES_COL] = valid["canonical_SMILES"]

    if len(valid) < N_SAMPLE:
        print(f"Skipped: only {len(valid):,} valid rows, need {N_SAMPLE:,}")
        return None

    sampled = valid.sample(n=N_SAMPLE, random_state=SAMPLE_SEED).reset_index(drop=True)
    sampled.to_csv(out_path, index=False)
    print(f"{len(df):,} rows, {len(valid):,} valid, {len(sampled):,} sampled -> {out_path}")
    return sampled


out_path = f"{os.path.splitext(CSV_PATH)[0]}_sampled{N_SAMPLE // 1000}k.csv"
df_sampled = process_file(CSV_PATH, out_path)

In [ ]:
# Filter generated molecules by Morgan environment coverage of a reference set

import os
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
from tqdm import tqdm

RDLogger.DisableLog("rdApp.*")

GEN_CSV = "FILE_PATH_HERE"
REF_CSV = "FILE_PATH_HERE"
GEN_SMILES_COL = "SMILES"
REF_SMILES_COL = "SMILES"
OUTPUT_DIR = "filtered_outputs"
RADIUS = 2
THRESHOLDS = [1.00, 0.90]

os.makedirs(OUTPUT_DIR, exist_ok=True)


def morgan_ids(mol):
    info = {}
    rdMolDescriptors.GetMorganFingerprint(mol, RADIUS, bitInfo=info)
    return set(info.keys())


ref_bits = set()
n_ref_valid = 0
for smi in tqdm(pd.read_csv(REF_CSV)[REF_SMILES_COL].dropna().astype(str), desc="reference"):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    n_ref_valid += 1
    ref_bits.update(morgan_ids(mol))
print(f"{n_ref_valid:,} valid reference molecules, {len(ref_bits):,} distinct Morgan environments")


def coverage(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.nan
    bits = morgan_ids(mol)
    if not bits:
        return np.nan
    return len(bits & ref_bits) / len(bits)


gen_df = pd.read_csv(GEN_CSV)
if GEN_SMILES_COL not in gen_df.columns:
    raise KeyError(f"Column '{GEN_SMILES_COL}' not found. Columns: {list(gen_df.columns)}")
n_input = len(gen_df)
gen_df["coverage"] = [coverage(s) for s in tqdm(gen_df[GEN_SMILES_COL].astype(str), desc="coverage")]
print(f"{n_input:,} generated molecules, {gen_df['coverage'].isna().sum():,} invalid")

base = os.path.splitext(os.path.basename(GEN_CSV))[0]
for t in sorted(THRESHOLDS, reverse=True):
    kept = gen_df[gen_df["coverage"] >= t]
    out_path = os.path.join(OUTPUT_DIR, f"{base}_coverage{round(t * 100)}.csv")
    kept.to_csv(out_path, index=False)
    frac = len(kept) / n_input if n_input else 0.0
    print(f"coverage >= {t:.2f}: {len(kept):,} of {n_input:,} ({frac:.3f}) -> {out_path}")

In [ ]:
# Diversity of generated hits

import os
import pandas as pd
from rdkit import Chem, RDLogger, SimDivFilters
from rdkit.Chem import rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

CSV_PATH = "FILE_PATH_HERE"
SMILES_COL = "SMILES"
SCORE_COL = "binary_activity"
SCORE_THRESH = 0.5
RADIUS = 2
N_BITS = 2048
CLUSTER_CUTOFF = 0.65  # Tanimoto distance


def get_hits(path):
    df = pd.read_csv(path)
    return df.loc[df[SCORE_COL] > SCORE_THRESH, SMILES_COL].dropna().astype(str).tolist()


def n_unique_substructures(smiles_list):
    keys, n_valid = set(), 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        n_valid += 1
        keys.update(rdMolDescriptors.GetMorganFingerprint(mol, radius=RADIUS).GetNonzeroElements())
    return len(keys), n_valid


def n_clusters(smiles_list, cutoff=CLUSTER_CUTOFF):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fps.append(rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=N_BITS))
    if not fps:
        return 0
    return len(SimDivFilters.LeaderPicker().LazyBitVectorPick(fps, len(fps), cutoff))


smis = get_hits(CSV_PATH)
n_uniq, n_valid = n_unique_substructures(smis)
metrics = {
    "file": os.path.basename(CSV_PATH),
    "n_hits": len(smis),
    "n_valid": n_valid,
    "n_unique_substructures": n_uniq,
    "n_clusters": n_clusters(smis),
}
print(pd.Series(metrics).to_string())

In [ ]:
# Fingerprint UMAP of generated hits vs known MRSA hits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
import umap.umap_ as umap

RDLogger.DisableLog("rdApp.*")

# (path, label, keep only hits, colour)
DATASETS = [
    ("FILE_PATH_HERE", "gCLM", True, "red"),
    ("FILE_PATH_HERE", "cCLM", True, "mediumpurple"),
    ("FILE_PATH_HERE", "MRSA hits", False, "#0000B0"),
]
HIGHLIGHT = "MRSA hits"  # drawn on top

SMILES_COL = "SMILES"
SCORE_COL = "binary_activity"
SCORE_THRESH = 0.5
SAMPLE_N = 2500
RANDOM_STATE = 42
RADIUS, N_BITS = 2, 2048

OUT_CSV = "fingerprint_umap_values.csv"
OUT_SVG = "fingerprint_umap.svg"


def smiles_to_fp(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=N_BITS)
    arr = np.zeros((N_BITS,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


frames = []
for path, label, filter_by_score, _ in DATASETS:
    df = pd.read_csv(path)
    if filter_by_score:
        df = df[df[SCORE_COL] > SCORE_THRESH]
    n_available = len(df)
    df = df.sample(n=min(SAMPLE_N, n_available), random_state=RANDOM_STATE)

    out = pd.DataFrame({
        "source": label,
        "input_path": path,
        "original_index": df.index,
        "SMILES": df[SMILES_COL].values,
        "binary_activity": df[SCORE_COL].values if SCORE_COL in df.columns else np.nan,
    })
    out["fp"] = [smiles_to_fp(s) for s in out["SMILES"]]
    out = out[out["fp"].map(lambda a: a is not None)]
    print(f"{label}: {n_available:,} available, {len(df):,} sampled, {len(out):,} valid")
    frames.append(out)

plot_df = pd.concat(frames, ignore_index=True)
X = np.vstack(plot_df.pop("fp").tolist())

emb = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric="jaccard",
                random_state=RANDOM_STATE).fit_transform(X)
plot_df["UMAP1"] = emb[:, 0]
plot_df["UMAP2"] = emb[:, 1]
plot_df.to_csv(OUT_CSV, index=False)

plt.figure(figsize=(8, 6))
for _, label, _, color in DATASETS:
    g = plot_df[plot_df["source"] == label]
    hi = label == HIGHLIGHT
    plt.scatter(g["UMAP1"], g["UMAP2"], s=7 if hi else 4, alpha=0.6 if hi else 0.4,
                zorder=3 if hi else 2, c=color, label=f"{label} (n={len(g)})")
plt.title(f"UMAP of Morgan fingerprints, score > {SCORE_THRESH}")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(frameon=False, markerscale=2)
plt.tight_layout()
plt.savefig(OUT_SVG)
plt.show()

In [ ]:
# Physicochemical property UMAP of generated hits vs training hits

import os
import sys
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDConfig, RDLogger
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem.GraphDescriptors import BertzCT
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap

RDLogger.DisableLog("rdApp.*")

# (path, label, keep only hits, colour)
DATASETS = [
    ("FILE_PATH_HERE", "gCLM", True, "red"),
    ("FILE_PATH_HERE", "cCLM", True, "mediumpurple"),
    ("FILE_PATH_HERE", "Training hits", False, "#0000B0"),
]
REFERENCE = "Training hits"  # not subsampled, drawn on top

SMILES_COL = "SMILES"
SCORE_COL = "binary_activity"
SCORE_THRESH = 0.5
SAMPLE_N = 2500
RANDOM_STATE = 42
OUT_PREFIX = "physchem_umap"

sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
sys.path.append(os.path.join(RDConfig.RDContribDir, "NP_Score"))
import sascorer
import npscorer

NP_MODEL = npscorer.readNPModel()


def pct_rotatable_bonds(m):
    nb = m.GetNumBonds()
    return Lipinski.NumRotatableBonds(m) / nb if nb else 0.0


def pct_stereocentres(m):
    nh = m.GetNumHeavyAtoms()
    if not nh:
        return 0.0
    return len(Chem.FindMolChiralCenters(m, includeUnassigned=True,
                                         useLegacyImplementation=False)) / nh


PROPS = {
    "MW": Descriptors.MolWt,
    "LogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "BertzTC": BertzCT,
    "QED": Descriptors.qed,
    "pct_sp3": Lipinski.FractionCSP3,
    "pct_rot_bonds": pct_rotatable_bonds,
    "n_rings": Lipinski.RingCount,
    "n_aliph_rings": Lipinski.NumAliphaticRings,
    "n_arom_rings": Lipinski.NumAromaticRings,
    "HBD": Lipinski.NumHDonors,
    "HBA": Lipinski.NumHAcceptors,
    "pct_stereo": pct_stereocentres,
    "SA_score": sascorer.calculateScore,
    "NP_score": lambda m: npscorer.scoreMol(m, NP_MODEL),
}

NICE = {"BertzTC": "Bertz CT", "pct_sp3": "fraction sp3", "pct_rot_bonds": "fraction rotatable bonds",
        "pct_stereo": "fraction stereocentres", "n_rings": "rings", "n_aliph_rings": "aliphatic rings",
        "n_arom_rings": "aromatic rings", "SA_score": "SA score", "NP_score": "NP-likeness",
        "LogP": "cLogP"}


def featurize(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    vals = [f(mol) for f in PROPS.values()]
    if not np.all(np.isfinite(vals)):
        return None
    return Chem.MolToSmiles(mol), vals


parts = []
for path, label, filter_by_score, _ in DATASETS:
    df = pd.read_csv(path)
    if filter_by_score:
        df = df[df[SCORE_COL] > SCORE_THRESH]
    n_available = len(df)
    if label != REFERENCE:
        df = df.sample(n=min(SAMPLE_N, n_available), random_state=RANDOM_STATE)

    rows = []
    for idx, smi, score in zip(df.index, df[SMILES_COL], df[SCORE_COL]):
        r = featurize(smi)
        if r is not None:
            rows.append((label, idx, smi, r[0], score, *r[1]))
    part = pd.DataFrame(rows, columns=["source", "original_index", "SMILES", "canonical",
                                       "binary_activity", *PROPS])
    print(f"{label}: {n_available:,} available, {len(df):,} sampled, {len(part):,} valid")
    parts.append(part)

plot_df = pd.concat(parts, ignore_index=True)
X = StandardScaler().fit_transform(plot_df[list(PROPS)].values)

emb = umap.UMAP(n_neighbors=50, min_dist=0.5, n_components=2, metric="euclidean",
                random_state=RANDOM_STATE).fit_transform(X)
plot_df["UMAP1"] = emb[:, 0]
plot_df["UMAP2"] = emb[:, 1]
plot_df.to_csv(f"{OUT_PREFIX}_values.csv", index=False)

plt.figure(figsize=(8, 6))
for _, label, _, color in DATASETS:
    g = plot_df[plot_df["source"] == label]
    hi = label == REFERENCE
    plt.scatter(g["UMAP1"], g["UMAP2"], s=7 if hi else 4, alpha=0.7 if hi else 0.35,
                zorder=3 if hi else 2, c=color, label=f"{label} (n={len(g):,})")
prop_list = textwrap.fill(", ".join(NICE.get(p, p) for p in PROPS), width=90)
plt.title(f"UMAP of molecular properties, score > {SCORE_THRESH}\n{prop_list}", fontsize=9)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(frameon=False, markerscale=2)
plt.tight_layout()
plt.savefig(f"{OUT_PREFIX}.svg")
plt.savefig(f"{OUT_PREFIX}.png", dpi=200)
plt.show()

In [ ]:
# Wasserstein distance per property: generated libraries vs reference set

import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import wasserstein_distance
from rdkit import Chem, RDConfig, RDLogger
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem.GraphDescriptors import BertzCT

RDLogger.DisableLog("rdApp.*")

sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
sys.path.append(os.path.join(RDConfig.RDContribDir, "NP_Score"))
import sascorer
import npscorer

NP_MODEL = npscorer.readNPModel()

REFERENCE_CSV = "FILE_PATH_HERE"
REFERENCE_FILTER = True
REFERENCE_LABEL = "the reference set"

LIBRARIES = {
    "cCLM": {
        "seed1": "FILE_PATH_HERE",
        "seed2": "FILE_PATH_HERE",
        "seed3": "FILE_PATH_HERE",
    },
    "gCLM06": {
        "seed22": "FILE_PATH_HERE",
        "seed32": "FILE_PATH_HERE",
        "seed42": "FILE_PATH_HERE",
    },
}

SMILES_COL, ACTIVITY_COL, ACTIVITY_MIN = "smiles", "binary_activity", 0.5
DRAW_N, N_REPS, SEED = 10_000, 1, 42

OUT_FIG = "wd_smallmultiples.svg"
OUT_CSV = "wd_per_run.csv"
OUT_SUMMARY_CSV = "wd_summary.csv"

STYLE = {"cCLM": ("#2a78d6", "o"),
         "gCLM06": ("#eb6834", "^"),
         "gCLM07": ("#8c3d1a", "v"),
         "cCLM y=0.8": ("#3f9b6d", "s")}
GREY = "#898781"
UNITS = {"MW": "Da", "TPSA": "Å²"}

def pct_rotatable_bonds(m):
    nb = m.GetNumBonds()
    return Lipinski.NumRotatableBonds(m) / nb if nb else 0.0


def pct_stereocentres(m):
    nh = m.GetNumHeavyAtoms()
    if not nh:
        return 0.0
    return len(Chem.FindMolChiralCenters(m, includeUnassigned=True,
                                         useLegacyImplementation=False)) / nh


PROPS = {
    "MW": Descriptors.MolWt,
    "LogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "BertzTC": BertzCT,
    "QED": Descriptors.qed,
    "pct_sp3": Lipinski.FractionCSP3,
    "pct_rot_bonds": pct_rotatable_bonds,
    "n_rings": Lipinski.RingCount,
    "n_aliph_rings": Lipinski.NumAliphaticRings,
    "n_arom_rings": Lipinski.NumAromaticRings,
    "HBD": Lipinski.NumHDonors,
    "HBA": Lipinski.NumHAcceptors,
    "pct_stereo": pct_stereocentres,
    "SA_score": sascorer.calculateScore,
    "NP_score": lambda m: npscorer.scoreMol(m, NP_MODEL),
}

NICE = {"BertzTC": "Bertz CT", "pct_sp3": "fraction sp3", "pct_rot_bonds": "fraction rotatable bonds",
        "pct_stereo": "fraction stereocentres", "n_rings": "rings", "n_aliph_rings": "aliphatic rings",
        "n_arom_rings": "aromatic rings", "SA_score": "SA score", "NP_score": "NP-likeness",
        "LogP": "cLogP"}


def read_smiles(path, label, activity_filter=False):
    df = pd.read_csv(path)
    col = [c for c in df.columns if c.strip().lower() == SMILES_COL.lower()][0]
    n_raw = len(df)
    if activity_filter and ACTIVITY_COL:
        df = df[pd.to_numeric(df[ACTIVITY_COL], errors="coerce") > ACTIVITY_MIN]
    smiles = df[col].dropna().astype(str).tolist()
    print(f"[{label:>18}] {n_raw:>7} rows -> {len(smiles):>7} hits")
    return smiles


def descriptors(smiles):
    rows = []
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            rows.append({k: float(f(mol)) for k, f in PROPS.items()})
    return pd.DataFrame(rows, columns=list(PROPS))


rng = np.random.default_rng(SEED)
ref = descriptors(read_smiles(REFERENCE_CSV, "reference", REFERENCE_FILTER))
print(f"reference: {len(ref):,} valid molecules; {N_REPS} draw(s) of {DRAW_N:,} per run")

gens = {}
for model, runs in LIBRARIES.items():
    gens[model] = {}
    for run_label, path in runs.items():
        smiles = read_smiles(path, f"{model}/{run_label}", activity_filter=True)
        m = min(DRAW_N, len(smiles))
        if m < DRAW_N:
            print(f"    only {len(smiles)} hits, drawing {m}")
        gens[model][run_label] = [
            descriptors([smiles[i] for i in rng.choice(len(smiles), m, replace=False)])
            for _ in range(N_REPS)]

rows = []
for prop in PROPS:
    r = ref[prop].dropna().to_numpy()
    for model, per in gens.items():
        for run_label, draws in per.items():
            d = [wasserstein_distance(r, dr[prop].dropna().to_numpy()) for dr in draws]
            rows.append({"property": prop, "library": model, "run": run_label,
                         "wasserstein": np.mean(d),
                         "sd_over_draws": np.std(d, ddof=1) if len(d) > 1 else np.nan,
                         "n_generated": int(np.mean([len(dr) for dr in draws])),
                         "n_reference": len(r)})
per_run = pd.DataFrame(rows)
per_run.to_csv(OUT_CSV, index=False)

summary = (per_run.groupby(["property", "library"], sort=False)["wasserstein"]
           .agg(mean="mean", sd="std", n_runs="count").reset_index())
summary.to_csv(OUT_SUMMARY_CSV, index=False)

print(per_run.pivot_table(index="property", columns=["library", "run"],
                          values="wasserstein", sort=False).round(3).to_string())
print()
print(summary.pivot_table(index="property", columns="library", values=["mean", "sd"],
                          sort=False).round(3).to_string())

libs = list(LIBRARIES)
order = list(PROPS)
vals = {p: {lib: per_run[(per_run.property == p) & (per_run.library == lib)]["wasserstein"].values
            for lib in libs} for p in order}

ncol = 5
nrow = int(np.ceil(len(order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.85 * ncol, 1.95 * nrow))
axes = np.atleast_1d(axes).ravel()
ypos = {lib: len(libs) - 1 - i for i, lib in enumerate(libs)}

for ax, p in zip(axes, order):
    v = vals[p]
    for lib in libs:
        col, mk = STYLE.get(lib, (GREY, "o"))
        x, yy = v[lib], ypos[lib]
        ax.scatter(x, np.full(len(x), yy), s=38, color=col, marker=mk,
                   edgecolors="white", linewidths=0.7, zorder=3)
        ax.plot([x.mean()] * 2, [yy - 0.26, yy + 0.26], color=col, lw=2.4, zorder=2)
    ax.set_xlim(0, max(x.max() for x in v.values()) * 1.18)
    ax.set_ylim(-0.7, len(libs) - 0.3)
    ax.set_yticks(list(ypos.values()))
    ax.set_yticklabels(list(ypos), fontsize=7.5)
    ax.set_title(NICE.get(p, p) + (f"  ({UNITS[p]})" if p in UNITS else ""), fontsize=10, pad=6)
    ax.tick_params(labelsize=7.5)
    ax.grid(alpha=0.13, axis="x", lw=0.6)
    ax.spines[["top", "right", "left"]].set_visible(False)

for ax in axes[len(order):]:
    ax.axis("off")

n_runs = {lib: len(vals[order[0]][lib]) for lib in libs}
fig.legend(handles=[Line2D([], [], marker=STYLE.get(lib, (GREY, "o"))[1], color="none",
                           markerfacecolor=STYLE.get(lib, (GREY,))[0], markersize=7.5,
                           label=f"{lib}  (n={n_runs[lib]} runs)") for lib in libs],
           fontsize=8.5, frameon=False, loc="lower right", ncol=len(libs))
fig.suptitle(f"Distance from generated libraries to {REFERENCE_LABEL}", fontsize=13, y=0.995)
fig.text(0.5, 0.935, f"Wasserstein distance per property, native units, "
         f"{DRAW_N:,}-molecule draws. Lower is closer.", fontsize=9.5, color=GREY, ha="center")
fig.text(0.006, 0.004, "Markers: individual runs. Bar: mean across runs.", fontsize=8.2, color=GREY)
fig.tight_layout(rect=[0, 0.04, 1, 0.92])
fig.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()